In [ ]:
!pip install cvss

In [ ]:
import json
import requests
import zipfile
import io
import re
from datetime import datetime, timezone
from cvss import CVSS2, CVSS3, CVSS4

In [ ]:
allCWE = {}
CWEmetricks = {}

In [ ]:
ECOSYSTEMS = {
    "PyPI": "https://osv-vulnerabilities.storage.googleapis.com/PyPI/all.zip",
    "npm": "https://osv-vulnerabilities.storage.googleapis.com/npm/all.zip",  # в OSV экосистема называется npm
    "Go": "https://osv-vulnerabilities.storage.googleapis.com/Go/all.zip"
}

def download_and_extract_zip(url):
    """Скачивает ZIP-архив и возвращает список JSON-объектов (записей об уязвимостях)"""
    print(f"Загрузка {url}...")
    response = requests.get(url)
    if response.status_code != 200:
        print(f"Ошибка загрузки {url}: {response.status_code}")
        return []

    vulnerabilities = []
    # Открываем ZIP из загруженных байтов
    with zipfile.ZipFile(io.BytesIO(response.content)) as z:
        for filename in z.namelist():
            if filename.endswith('.json'):
                with z.open(filename) as f:
                    try:
                        vuln = json.load(f)
                        vulnerabilities.append(vuln)
                    except json.JSONDecodeError:
                        print(f"Ошибка чтения JSON из файла {filename}")
    return vulnerabilities

In [ ]:
pypi_vulnerabilities = download_and_extract_zip(ECOSYSTEMS['PyPI'])
npm_vulnerabilities = download_and_extract_zip(ECOSYSTEMS['npm'])
go_vulnerabilities = download_and_extract_zip(ECOSYSTEMS['Go'])

Загрузка https://osv-vulnerabilities.storage.googleapis.com/PyPI/all.zip...
Загрузка https://osv-vulnerabilities.storage.googleapis.com/npm/all.zip...
Загрузка https://osv-vulnerabilities.storage.googleapis.com/Go/all.zip...


In [ ]:
for i in range(100):
  print(pypi_vulnerabilities[i]["database_specific"])

{'cwe_ids': ['CWE-20', 'CWE-22', 'CWE-73'], 'github_reviewed': True, 'github_reviewed_at': '2025-03-21T16:32:50Z', 'nvd_published_at': '2025-03-20T10:15:23Z', 'severity': 'CRITICAL'}
{'cwe_ids': ['CWE-94'], 'github_reviewed': True, 'github_reviewed_at': '2025-05-20T18:01:52Z', 'nvd_published_at': '2025-05-20T18:15:46Z', 'severity': 'HIGH'}
{'cwe_ids': [], 'github_reviewed': True, 'github_reviewed_at': '2024-02-28T22:58:40Z', 'nvd_published_at': '2024-02-28T20:15:41Z', 'severity': 'MODERATE'}
{'cwe_ids': ['CWE-434', 'CWE-669'], 'github_reviewed': True, 'github_reviewed_at': '2025-04-18T20:24:07Z', 'nvd_published_at': None, 'severity': 'HIGH'}
{'cwe_ids': ['CWE-444'], 'github_reviewed': True, 'github_reviewed_at': '2021-09-17T18:30:53Z', 'nvd_published_at': '2021-09-16T15:15:00Z', 'severity': 'CRITICAL'}
{'cwe_ids': ['CWE-20'], 'github_reviewed': True, 'github_reviewed_at': '2023-07-17T22:02:54Z', 'nvd_published_at': '2019-08-22T17:15:00Z', 'severity': 'MODERATE'}
{'cwe_ids': ['CWE-200']

In [ ]:
ECOSYSTEM_archive = {"PyPI": pypi_vulnerabilities,"npm": npm_vulnerabilities, "Go" : go_vulnerabilities}

In [ ]:
pypi_vulnerabilities

In [75]:
for x in pypi_vulnerabilities:
  if "upstream" in x:
    if len(x["upstream"]) > 0:
      print(1)

In [ ]:
def extract_cvss_severity(vuln):
    """Извлекает числовое значение из CVSS score"""
    severity_list = vuln.get("severity", [])
    for severity in severity_list:
        score_str = severity.get("score")
        match = re.search(r"(\d+\.?\d*)", score_str)
        if not match:
          continue
        if severity.get("type") == "CVSS_V4":
            c = CVSS4(severity.get("score"))
            return c.scores()[0]
        elif severity.get("type") == "CVSS_V3":
            c = CVSS3(severity.get("score"))
            return c.scores()[0]
        else:
            c = CVSS2(severity.get("score"))
            return c.scores()[0]
    return None

def calculate_fix_time_days(vuln):
    """Вычисляет время исправления в днях (разница между modified и published)"""
    published = vuln.get("published")
    modified = vuln.get("modified")

    if not published or not modified:
        return None
    try:
        # Приводим даты к UTC для единообразия
        pub_date = datetime.fromisoformat(published.replace('Z', '+00:00'))
        mod_date = datetime.fromisoformat(modified.replace('Z', '+00:00'))

        # Вычисляем разницу в днях
        delta = mod_date - pub_date
        return delta.days
    except ValueError:
        return None

def process_ecosystem(ecosystem_name, url):
    """Основная функция для обработки одной экосистемы"""
    print(f"\n--- Обработка экосистемы: {ecosystem_name} ---")
    results = []

    # 1. Загружаем все уязвимости
    vulnerabilities = ECOSYSTEM_archive[ecosystem_name]
    total = len(vulnerabilities)
    print(f"Загружено {total} записей")

    # 2. Фильтруем и извлекаем данные
    for i, vuln in enumerate(vulnerabilities):
        if (i + 1) % 1000 == 0:
            print(f"Обработано {i+1}/{total} записей...")

        # Проверяем наличие severity
        severity_score = extract_cvss_severity(vuln)
        if severity_score is None:
            continue
        fix_time = calculate_fix_time_days(vuln)
        if "database_specific" in vuln:
          if "cwe_ids" in vuln["database_specific"]:
              for cwe in vuln["database_specific"]["cwe_ids"]:
                if cwe not in allCWE:
                  allCWE[cwe] = 0
                if cwe not in CWEmetricks:
                  CWEmetricks[cwe] = 0
                allCWE[cwe] += 1
                CWEmetricks[cwe] += fix_time * severity_score

        results.append({
            "id": vuln.get("id"),
            "ecosystem": ecosystem_name,
            "summary": vuln.get("summary", ""),
            "cvss_v3_score": severity_score,
            "fix_time_days": fix_time,
            "published": vuln.get("published"),
            "modified": vuln.get("modified"),
            "cwe_ids": vuln.get("database_specific",[])
        })

    print(f"Найдено записей с severity: {len(results)}")
    return results

if __name__ == "__main__":
    all_results = []

    # Обрабатываем каждую экосистему
    for name, url in ECOSYSTEMS.items():
        ecosystem_results = process_ecosystem(name, url)
        all_results.extend(ecosystem_results)

    # Сохраняем результат
    with open('vulnerabilities_analysis.json', 'w', encoding='utf-8') as f:
        json.dump(all_results, f, indent=2, ensure_ascii=False)

    print(f"\nГотово! Всего обработано записей с severity: {len(all_results)}")
    print("Результат сохранен в файл 'vulnerabilities_analysis.json'")


--- Обработка экосистемы: PyPI ---
Загружено 19564 записей
Обработано 1000/19564 записей...
Обработано 2000/19564 записей...
Обработано 3000/19564 записей...
Обработано 4000/19564 записей...
Обработано 5000/19564 записей...
Обработано 6000/19564 записей...
Обработано 7000/19564 записей...
Обработано 8000/19564 записей...
Обработано 9000/19564 записей...
Обработано 10000/19564 записей...
Обработано 11000/19564 записей...
Обработано 12000/19564 записей...
Обработано 13000/19564 записей...
Обработано 14000/19564 записей...
Обработано 15000/19564 записей...
Обработано 16000/19564 записей...
Обработано 17000/19564 записей...
Обработано 18000/19564 записей...
Обработано 19000/19564 записей...
Найдено записей с severity: 5419

--- Обработка экосистемы: npm ---
Загружено 218681 записей
Обработано 1000/218681 записей...
Обработано 2000/218681 записей...
Обработано 3000/218681 записей...
Обработано 4000/218681 записей...
Обработано 5000/218681 записей...
Обработано 6000/218681 записей...
Обрабо

In [ ]:
allCWE

{'CWE-20': 933,
 'CWE-22': 1358,
 'CWE-73': 93,
 'CWE-94': 643,
 'CWE-434': 126,
 'CWE-669': 20,
 'CWE-444': 86,
 'CWE-200': 784,
 'CWE-79': 1790,
 'CWE-754': 86,
 'CWE-601': 336,
 'CWE-640': 15,
 'CWE-502': 413,
 'CWE-674': 96,
 'CWE-287': 384,
 'CWE-787': 238,
 'CWE-284': 375,
 'CWE-125': 260,
 'CWE-552': 42,
 'CWE-791': 13,
 'CWE-362': 121,
 'CWE-367': 91,
 'CWE-1270': 3,
 'CWE-617': 139,
 'CWE-400': 889,
 'CWE-918': 577,
 'CWE-611': 89,
 'CWE-190': 136,
 'CWE-409': 53,
 'CWE-704': 19,
 'CWE-755': 67,
 'CWE-789': 37,
 'CWE-777': 3,
 'CWE-193': 15,
 'CWE-416': 48,
 'CWE-122': 50,
 'CWE-323': 6,
 'CWE-693': 92,
 'CWE-306': 208,
 'CWE-78': 542,
 'CWE-639': 187,
 'CWE-862': 309,
 'CWE-369': 129,
 'CWE-250': 31,
 'CWE-311': 89,
 'CWE-322': 7,
 'CWE-770': 513,
 'CWE-77': 346,
 'CWE-772': 21,
 'CWE-319': 36,
 'CWE-377': 30,
 'CWE-863': 629,
 'CWE-201': 40,
 'CWE-209': 79,
 'CWE-67': 6,
 'CWE-424': 3,
 'CWE-532': 188,
 'CWE-185': 25,
 'CWE-59': 157,
 'CWE-345': 142,
 'CWE-426': 44,
 'CWE-82

In [ ]:
CWEmetricks

In [ ]:
import pandas as pd

In [ ]:
for cwe, count in allCWE.items():
  CWEmetricks[cwe] = CWEmetricks[cwe] / count

In [ ]:
firstCOL = "Насколько быстро экосистема экосистемы реагируют на данный тип проблемы"

In [ ]:
secondCOL = "Насколько быстро экосистема экосистемы реагируют на данный тип проблемы"

In [ ]:
data = pd.DataFrame({
    firstCOL : allCWE,
    secondCOL : CWEmetricks
    })

In [ ]:
data

,Насколько быстро экосистема экосистемы реагируют на данный тип проблемы
CWE-20,5571.233655
CWE-22,3387.350589
CWE-73,924.546237
CWE-94,4007.132193
CWE-434,3118.023016
...,...
CWE-566,1749.600000
CWE-1390,90.200000
CWE-549,1330.300000
CWE-495,58.500000


In [ ]:
df_sorted = data.sort_values(by=secondCOL, ascending=False)

In [71]:
upstream_graph = {}   # Формат: { "ID": ["Upstream_ID_1", "Upstream_ID_2"] }
ecosystem_map = {}    # Формат: { "ID": "PyPI" } (чтобы знать, к кому относится)

In [73]:
import json
import requests
import zipfile
import io
import re
import pandas as pd
from datetime import datetime, timezone
from cvss import CVSS2, CVSS3, CVSS4

ECOSYSTEMS = {
    "PyPI": "https://osv-vulnerabilities.storage.googleapis.com/PyPI/all.zip",
    "npm": "https://osv-vulnerabilities.storage.googleapis.com/npm/all.zip",
    "Go": "https://osv-vulnerabilities.storage.googleapis.com/Go/all.zip"
}

def extract_cvss_severity(vuln):
    """Извлекает числовое значение из CVSS score"""
    severity_list = vuln.get("severity", [])
    for severity in severity_list:
        score_str = severity.get("score")
        match = re.search(r"(\d+\.?\d*)", score_str)
        if not match:
            continue
        if severity.get("type") == "CVSS_V4":
            c = CVSS4(severity.get("score"))
            return c.scores()[0]
        elif severity.get("type") == "CVSS_V3":
            c = CVSS3(severity.get("score"))
            return c.scores()[0]
        else:
            c = CVSS2(severity.get("score"))
            return c.scores()[0]
    return None

def calculate_fix_time_days(vuln):
    """Вычисляет время исправления в днях (разница между modified и published)"""
    published = vuln.get("published")
    modified = vuln.get("modified")

    if not published or not modified:
        return None
    try:
        pub_date = datetime.fromisoformat(published.replace('Z', '+00:00'))
        mod_date = datetime.fromisoformat(modified.replace('Z', '+00:00'))

        delta = mod_date - pub_date
        return delta.days
    except ValueError:
        return None

def process_ecosystem(ecosystem_name, url):
    """Основная функция для обработки одной экосистемы"""
    print(f"\n--- Обработка экосистемы: {ecosystem_name} ---")
    results = []

    # Словари теперь локальные для каждой экосистемы
    ecosystem_cwe_count = {}
    ecosystem_cwe_metric = {}

    # 1. Скачиваем и распаковываем данные в памяти
    print(f"Скачивание архива для {ecosystem_name}...")
    response = requests.get(url)
    vulnerabilities = []
    with zipfile.ZipFile(io.BytesIO(response.content)) as z:
        for filename in z.namelist():
            if filename.endswith('.json'):
                with z.open(filename) as f:
                    vulnerabilities.append(json.load(f))

    total = len(vulnerabilities)
    print(f"Загружено {total} записей")

    # 2. Фильтруем и извлекаем данные
    for i, vuln in enumerate(vulnerabilities):
        if (i + 1) % 1000 == 0:
            print(f"Обработано {i+1}/{total} записей...")

        vuln_id = vuln.get("id")
        ecosystems = ecosystem_name
        ecosystem_map[vuln_id] = ecosystem_name

        upstreams = vuln.get("upstream", [])
        upstream_graph[vuln_id] = upstreams

        severity_score = extract_cvss_severity(vuln)
        if severity_score is None:
            continue

        fix_time = calculate_fix_time_days(vuln)

        # Обновляем словари только для текущей экосистемы
        if "database_specific" in vuln and "cwe_ids" in vuln["database_specific"]:
            for cwe in vuln["database_specific"]["cwe_ids"]:
                if cwe not in ecosystem_cwe_count:
                    ecosystem_cwe_count[cwe] = 0
                    ecosystem_cwe_metric[cwe] = 0

                ecosystem_cwe_count[cwe] += 1

                # Защита от ошибки, если fix_time == None
                if fix_time is not None:
                    ecosystem_cwe_metric[cwe] += fix_time * severity_score

        results.append({
            "id": vuln.get("id"),
            "ecosystem": ecosystem_name,
            "summary": vuln.get("summary", ""),
            "cvss_v3_score": severity_score,
            "fix_time_days": fix_time,
            "published": vuln.get("published"),
            "modified": vuln.get("modified"),
            "cwe_ids": vuln.get("database_specific", {}).get("cwe_ids", [])
        })

    print(f"Найдено записей с severity: {len(results)}")
    # Возвращаем результаты и словари для пандаса
    return results, ecosystem_cwe_count, ecosystem_cwe_metric
if __name__ == "__main__":
    all_results = []

    # Словари для сбора данных под pandas
    all_counts_for_df = {}
    all_metrics_for_df = {}

    for name, url in ECOSYSTEMS.items():
        ecosystem_results, count_dict, metric_dict = process_ecosystem(name, url)
        all_results.extend(ecosystem_results)
        for cwe, count in count_dict.items():
          metric_dict[cwe] /= count
        # Сохраняем словари, задавая названия будущих колонок
        all_counts_for_df[f"{name}_Count"] = count_dict
        all_metrics_for_df[f"{name}_Metric"] = metric_dict

    # Сохраняем сырые json результаты
    # with open('vulnerabilities_analysis.json', 'w', encoding='utf-8') as f:
    #     json.dump(all_results, f, indent=2, ensure_ascii=False)

    # --- Блок Pandas ---
    print("\n--- Формирование pandas таблицы ---")
    # Создаем два DataFrame из собранных словарей
    df_counts = pd.DataFrame(all_counts_for_df)
    df_metrics = pd.DataFrame(all_metrics_for_df)

    # Объединяем по индексу (CWE). axis=1 означает соединение столбцов.
    # Если в какой-то экосистеме нет конкретного CWE, ставим 0
    df_final = pd.concat([df_counts, df_metrics], axis=1).fillna(0)

    # Опционально: сортируем столбцы для красоты (PyPI_Count, PyPI_Metric, npm_Count...)
    columns_order = []
    for name in ECOSYSTEMS.keys():
        columns_order.extend([f"{name}_Count", f"{name}_Metric"])
    df_final = df_final[columns_order]

    # Выводим первые строки и сохраняем в CSV
    print(df_final.head())
    df_final.to_csv("cwe_ecosystems_summary.csv")
    print("\nГотово! Статистика по CWE сохранена в 'cwe_ecosystems_summary.csv'")


--- Обработка экосистемы: PyPI ---
Скачивание архива для PyPI...
Загружено 19564 записей
Обработано 1000/19564 записей...
Обработано 2000/19564 записей...
Обработано 3000/19564 записей...
Обработано 4000/19564 записей...
Обработано 5000/19564 записей...
Обработано 6000/19564 записей...
Обработано 7000/19564 записей...
Обработано 8000/19564 записей...
Обработано 9000/19564 записей...
Обработано 10000/19564 записей...
Обработано 11000/19564 записей...
Обработано 12000/19564 записей...
Обработано 13000/19564 записей...
Обработано 14000/19564 записей...
Обработано 15000/19564 записей...
Обработано 16000/19564 записей...
Обработано 17000/19564 записей...
Обработано 18000/19564 записей...
Обработано 19000/19564 записей...
Найдено записей с severity: 5419

--- Обработка экосистемы: npm ---
Скачивание архива для npm...
Загружено 218692 записей
Обработано 1000/218692 записей...
Обработано 2000/218692 записей...
Обработано 3000/218692 записей...
Обработано 4000/218692 записей...
Обработано 5000

In [ ]:
df_pypi_sorted_by_count = df_final.sort_values(by="PyPI_Count", ascending=False)

In [ ]:
df_npm_sorted_by_count = df_final.sort_values(by="npm_Count", ascending=False)

In [ ]:
df_go_sorted_by_count = df_final.sort_values(by="Go_Count", ascending=False)

In [ ]:
df_pypi_sorted_by_count.head(15)["PyPI_Count"]

,PyPI_Count
CWE-79,441.0
CWE-22,348.0
CWE-20,280.0
CWE-200,210.0
CWE-400,198.0
CWE-94,190.0
CWE-502,188.0
CWE-918,147.0
CWE-770,130.0
CWE-787,109.0


In [ ]:
import csv
import io
import requests

def load_cwe_dictionary():
    """Скачивает справочник CWE и возвращает словарь для маппинга"""
    print("Загрузка официального справочника CWE от MITRE...")
    # View 1000 - это самый полный список (Research Concepts)
    url = "https://cwe.mitre.org/data/csv/1000.csv.zip"

    response = requests.get(url)
    cwe_dict = {}

    with zipfile.ZipFile(io.BytesIO(response.content)) as z:
        # Внутри архива лежит файл 1000.csv
        with z.open('1000.csv') as f:
            # Читаем CSV
            csv_data = f.read().decode('utf-8')
            reader = csv.reader(io.StringIO(csv_data))

            headers = next(reader)
            # Индексы колонок в CSV: 0 - ID, 1 - Name, 2 - Abstraction (уровень)
            for row in reader:
                if len(row) > 2:
                    cwe_id = f"CWE-{row[0]}"
                    cwe_name = row[1]
                    cwe_dict[cwe_id] = cwe_name

    print(f"Загружено {len(cwe_dict)} описаний CWE.")
    return cwe_dict

In [ ]:
cwe_dict = load_cwe_dictionary()

Загрузка официального справочника CWE от MITRE...
Загружено 944 описаний CWE.


In [ ]:
cwe_dict

{'CWE-5': 'J2EE Misconfiguration: Data Transmission Without Encryption',
 'CWE-6': 'J2EE Misconfiguration: Insufficient Session-ID Length',
 'CWE-7': 'J2EE Misconfiguration: Missing Custom Error Page',
 'CWE-8': 'J2EE Misconfiguration: Entity Bean Declared Remote',
 'CWE-9': 'J2EE Misconfiguration: Weak Access Permissions for EJB Methods',
 'CWE-11': 'ASP.NET Misconfiguration: Creating Debug Binary',
 'CWE-12': 'ASP.NET Misconfiguration: Missing Custom Error Page',
 'CWE-13': 'ASP.NET Misconfiguration: Password in Configuration File',
 'CWE-14': 'Compiler Removal of Code to Clear Buffers',
 'CWE-15': 'External Control of System or Configuration Setting',
 'CWE-20': 'Improper Input Validation',
 'CWE-22': "Improper Limitation of a Pathname to a Restricted Directory ('Path Traversal')",
 'CWE-23': 'Relative Path Traversal',
 'CWE-24': "Path Traversal: '../filedir'",
 'CWE-25': "Path Traversal: '/../filedir'",
 'CWE-26': "Path Traversal: '/dir/../filename'",
 'CWE-27': "Path Traversal: 'd

In [ ]:
df_pypi_sorted_by_count.head(15)["PyPI_Count"]

,PyPI_Count
CWE-79,441.0
CWE-22,348.0
CWE-20,280.0
CWE-200,210.0
CWE-400,198.0
CWE-94,190.0
CWE-502,188.0
CWE-918,147.0
CWE-770,130.0
CWE-787,109.0


In [ ]:
most_py_cwes = df_pypi_sorted_by_count.head(15)["PyPI_Count"].index

In [ ]:
most_npm_cwes = df_npm_sorted_by_count.head(15)["npm_Count"].index

In [ ]:
most_go_cwes = df_go_sorted_by_count.head(15)["Go_Count"].index

In [ ]:
if __name__ == "__main__":
    for name in ECOSYSTEMS.keys():
        metric_col = f"{name}_Metric"

        # Проверяем, что колонка существует (на случай, если экосистема была пустой)
        if metric_col in df_final.columns:
            # Получаем 10 самых больших значений из колонки метрики
            top_10 = df_final[metric_col].nlargest(20)

            print(f"\n🚀 Экосистема: {name}")
            print("-" * 35)

            # Выводим с нумерацией
            for rank, (cwe, score) in enumerate(top_10.items(), start=1):
                # Если метрика равна 0, значит значимых уязвимостей больше нет
                if score == 0:
                    break

                # Достаем количество (Count) для полноты картины
                count_col = f"{name}_Count"
                cwe_count = int(df_final.at[cwe, count_col])

                # Форматированный вывод (выравнивание пробелами для красоты)
                print(f"{rank:2d}. {cwe:<10} | Метрика: {score:10,.2f} | С какой областью связана уязвимость: {cwe_dict[cwe]} | Кол-во: {cwe_count}")

    print("\n" + "="*50)


🚀 Экосистема: PyPI
-----------------------------------
 1. CWE-242    | Метрика:  20,354.60 | С какой областью связана уязвимость: Use of Inherently Dangerous Function | Кол-во: 1
 2. CWE-197    | Метрика:  17,955.00 | С какой областью связана уязвимость: Numeric Truncation Error | Кол-во: 1
 3. CWE-336    | Метрика:  17,025.00 | С какой областью связана уязвимость: Same Seed in Pseudo-Random Number Generator (PRNG) | Кол-во: 1
 4. CWE-1056   | Метрика:  16,032.80 | С какой областью связана уязвимость: Invokable Control Element with Variadic Parameters | Кол-во: 1
 5. CWE-641    | Метрика:  14,106.40 | С какой областью связана уязвимость: Improper Restriction of Names for Files and Other Resources | Кол-во: 1
 6. CWE-391    | Метрика:  13,567.50 | С какой областью связана уязвимость: Unchecked Error Condition | Кол-во: 1
 7. CWE-527    | Метрика:  12,253.60 | С какой областью связана уязвимость: Exposure of Version-Control Repository to an Unauthorized Control Sphere | Кол-во: 1
 8. C

In [72]:
def find_longest_upstream_chains(graph, eco_map):
    print("\n--- Анализ графа Upstream ---")
    memo = {}        # Кэш длин: { "ID": длина }
    path_memo = {}   # Кэш самих путей: { "ID": ["ID1", "ID2", ...] }

    def dfs(node):
        # Если уже считали этот узел, просто возвращаем результат
        if node in memo:
            return memo[node], path_memo[node]

        # Если у узла нет upstream (или его нет в нашем графе), длина 1
        if node not in graph or not graph[node]:
            return 1, [node]

        max_len = 0
        best_path = []

        # Проверяем все ветки upstream и ищем самую длинную
        for neighbor in graph[node]:
            length, path = dfs(neighbor)
            if length > max_len:
                max_len = length
                best_path = path

        # Сохраняем результат: свой узел + лучший путь из соседей
        memo[node] = max_len + 1
        path_memo[node] = [node] + best_path

        return memo[node], path_memo[node]

    # 1. Считаем длины для всех узлов
    for node in graph.keys():
        dfs(node)

    # 2. Группируем рекорды по экосистемам
    longest_by_eco = {eco: {"length": 0, "path": []} for eco in set(eco_map.values())}

    for node, length in memo.items():
        eco = eco_map.get(node)
        if not eco:
            continue # Если узел "внешний", пропускаем

        if length > longest_by_eco[eco]["length"]:
            longest_by_eco[eco]["length"] = length
            longest_by_eco[eco]["path"] = path_memo[node]

    # 3. Выводим результаты
    for eco, data in longest_by_eco.items():
        print(f"\n🌐 Экосистема: {eco}")
        if data['length'] > 1:
            print(f"Самая длинная цепочка: {data['length']} узлов")
            print(" -> ".join(data['path']))
        else:
            print("В этой экосистеме нет иерархии upstream (все цепочки длиной 1).")

    return longest_by_eco

In [74]:
ans = find_longest_upstream_chains(upstream_graph, ecosystem_map)


--- Анализ графа Upstream ---

🌐 Экосистема: npm
В этой экосистеме нет иерархии upstream (все цепочки длиной 1).

🌐 Экосистема: PyPI
В этой экосистеме нет иерархии upstream (все цепочки длиной 1).

🌐 Экосистема: Go
В этой экосистеме нет иерархии upstream (все цепочки длиной 1).
